# 24 — Prompt Versioning, Experimentation, and Release Engineering

    ## Scenario and success criteria

    A candidate is routed by a stable request key and rolled back only with enough evidence—or immediately after a critical failure.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Use deterministic, sticky assignment.
- Separate insufficient samples from success.
- Make critical rollback conditions explicit.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 24 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Random per-request assignment breaks user consistency and makes incident reconstruction difficult.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab24 import CanaryEvidence, route_version, should_rollback

assignments = {request_id: route_version(request_id, canary_percent=20) for request_id in [f"R-{i}" for i in range(30)]}
print(assignments)

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
early = CanaryEvidence(requests=3, failures=1, critical_failures=0)
mature = CanaryEvidence(requests=100, failures=8, critical_failures=0)
critical = CanaryEvidence(requests=1, failures=1, critical_failures=1)
for evidence in (early, mature, critical):
    print(evidence, should_rollback(evidence))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert route_version("R-1", canary_percent=20) == route_version("R-1", canary_percent=20)
assert should_rollback(early)[1] == "insufficient_samples"
assert should_rollback(mature) == (True, "failure_rate")
assert should_rollback(critical) == (True, "critical_failure")

## Production upgrade

Pre-register metrics and guardrails, preserve cohort assignment, compare the same slices, define minimum samples and stop rules, and keep a tested stable artifact ready for rollback.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.